<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/07_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 · Memory: AGENTS.md, memory files, and learning across sessions

"Give the agent memory" sounds like one feature. It is **three different mechanisms**, with
three different owners and three different update cycles — and mixing them up is why so many
agents either forget everything or remember the wrong things forever.

| Tier | Who writes it | Changes how often | Lives where |
|---|---|---|---|
| **System prompt** | an engineer | at deploy time | your code |
| **`AGENTS.md`** | anyone on the team | any time, no deploy | the backend |
| **`/memories/`** | **the agent itself** | continuously | the store |

**New in this lesson:** `memory=[...]`, store-backed `/memories/`, namespaces, per-user scoping.

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-07-memory"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

---

## 1. Tier 1 — the system prompt

You have used this since lesson 01. It ships with your code, it is identical for every user,
and changing it needs an engineer and a deployment.

Right for: what the agent **is**. Wrong for: anything that varies by team, customer, or week.

---

## 2. Tier 2 — `AGENTS.md`

An `AGENTS.md` file is read from the backend at startup and folded into the system prompt. It
is the knob you can hand to someone who does not write Python.

In [ ]:
!mkdir -p agent_home/memory

In [ ]:
%%writefile agent_home/memory/AGENTS.md
# House style

- Address the customer by first name only, never "Dear Sir/Madam".
- Never promise a delivery date. Say "typically 3-5 business days".
- Every reply ends with the ticket reference on its own line, formatted: `Ref: T-####`
- If a refund is involved, always state the policy line you relied on.
- British spelling.

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

agent = create_deep_agent(
    model=MODEL,
    system_prompt="You are a customer support agent for an office furniture retailer.",
    backend=FilesystemBackend(root_dir="agent_home"),
    memory=["/memory/AGENTS.md"],
)

reply = agent.invoke({"messages": [{"role": "user", "content":
    "Draft a reply to Avery for ticket T-1: their standing desk arrived with a cracked leg."
}]})
print(reply["messages"][-1].text)

No code told it to sign off with a ref, or to use British spelling. Now change the rules
**without touching a line of Python**.

In [ ]:
%%writefile agent_home/memory/AGENTS.md
# House style

- Address the customer formally: "Dear Avery".
- Always give an explicit next step with a named owner.
- Do not include ticket references in the body; they are added by the ticketing system.
- Keep replies under 60 words.
- American spelling.

In [ ]:
# Same agent construction; only the file changed.
agent = create_deep_agent(
    model=MODEL,
    system_prompt="You are a customer support agent for an office furniture retailer.",
    backend=FilesystemBackend(root_dir="agent_home"),
    memory=["/memory/AGENTS.md"],
)

print(agent.invoke({"messages": [{"role": "user", "content":
    "Draft a reply to Avery for ticket T-1: their standing desk arrived with a cracked leg."
}]})["messages"][-1].text)

### 🧠 Checkpoint

Both tiers end up in the same system prompt. So why put house style in `AGENTS.md` rather than
in the `system_prompt` string?

Name one rule that belongs in each.

<details><summary>Show answer</summary>

Because they have **different owners and different release cycles**.

`AGENTS.md` is a file in the backend. A support lead can edit it at 4pm on a Friday because the
tone was wrong, with no pull request, no deploy, and no engineer. The system prompt is code:
changing it means a review and a release.

- **System prompt:** *"You are a customer support agent for an office furniture retailer."*
  This is what the agent fundamentally is. If it changes, you have a different product.
- **`AGENTS.md`:** *"Keep replies under 60 words. American spelling."* Policy and taste — the
  things that get revised constantly by people who are not engineers.

The test: **would a non-engineer reasonably want to change this without asking you?** If yes,
it belongs in `AGENTS.md`.

</details>

---

## 3. Tier 3 — memory the agent writes itself

The first two tiers are things *you* tell the agent. This one is what the agent learns and
records on its own, so a later conversation can use it.

There is no special machinery here. The agent already has filesystem tools from lesson 02; you
route `/memories/` to a store so what it writes outlives the thread, and use the system prompt
to say when to write and when to look.

In [ ]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

learning_agent = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You are a helpful assistant for a support team.\n"
        "At the START of every conversation, list /memories/ and read what is there.\n"
        "When you learn a durable preference or fact about how this person works, "
        "record it under /memories/ so future conversations can use it."
    ),
    backend=CompositeBackend(
        default=StateBackend(),
        routes={"/memories/": StoreBackend(
            namespace=lambda rt: ("workshop", "memories"), store=store)},
    ),
)

learning_agent.invoke({"messages": [{"role": "user", "content":
    "For future reference: I always want refund amounts shown in both USD and GBP, "
    "and I hate bullet points — write in short paragraphs."
}]})
print("done")

In [ ]:
# Inspect what it decided to keep. This is the bit people skip, and it is the bit that bites.
for item in store.search(("workshop", "memories")):
    print(f"--- {item.key} ---")
    print(str(item.value)[:400])
    print()

### The payoff

A brand-new conversation. Nothing carried over except the store.

In [ ]:
recalled = learning_agent.invoke({"messages": [{"role": "user", "content":
    "Order 1042 was refunded $429. Write me a one-line summary."
}]})
print(recalled["messages"][-1].text)

### 🧠 Checkpoint

Open the second run's trace. The preference was not in the messages you sent.

At what point did it enter the model's context — and what does that tell you about what
"recall" actually is?

<details><summary>Show answer</summary>

It entered **before the first model call**, when the memory middleware read `/memories/` from
the store and folded the contents into the system prompt.

So "recall" is not the model remembering anything. The model is stateless and has no idea a
previous conversation happened. Recall is **retrieval plus prompt assembly**: something read a
file and pasted it in front of the conversation.

That reframing is useful because it tells you where to debug. If the agent "forgot", the bug is
in what was written, what was retrieved, or what was assembled — three concrete, inspectable
places. None of them are inside the model.

</details>

---

## 4. Namespaces: keeping users apart

The `namespace` callable receives the runtime, so memory can be scoped per user, per team, per
tenant. Get this wrong and you have built a data leak.

In [ ]:
from dataclasses import dataclass


@dataclass
class Context:
    user_id: str


def per_user(rt) -> tuple[str, ...]:
    # Scope every memory read and write to the current user.
    return ("users", rt.context.user_id, "memories")


scoped_agent = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You are a helpful assistant. At the start of every conversation, list /memories/ "
        "and read what is there. Record durable user preferences under /memories/."
    ),
    backend=CompositeBackend(
        default=StateBackend(),
        routes={"/memories/": StoreBackend(namespace=per_user, store=store)},
    ),
    context_schema=Context,
)

scoped_agent.invoke(
    {"messages": [{"role": "user", "content":
        "Remember: my department code is ENG-4471 and I approve refunds up to $500."}]},
    context=Context(user_id="avery"),
)

print("--- what avery sees ---")
print(scoped_agent.invoke(
    {"messages": [{"role": "user", "content": "What is my department code?"}]},
    context=Context(user_id="avery"),
)["messages"][-1].text[:200])

print("\n--- what jordan sees ---")
print(scoped_agent.invoke(
    {"messages": [{"role": "user", "content": "What is my department code?"}]},
    context=Context(user_id="jordan"),
)["messages"][-1].text[:200])

In [ ]:
# The namespaces really are separate keys in the store.
for ns in [("users", "avery", "memories"), ("users", "jordan", "memories")]:
    items = list(store.search(ns))
    print(f"{ns}: {len(items)} item(s)")

---

## 5. How agent-written memory goes wrong

Three failure modes, all of which appear only after weeks in production:

**1. Hoarding.** Everything looks worth remembering, so `/memories/` grows without bound. It is
loaded into the prompt on every call, so your cost per request climbs forever and quality drops
as signal thins out. *Fix:* tell the agent what qualifies, and prune on a schedule.

**2. Contradiction.** In March the user prefers bullet points; in June they do not. Both notes
are in the store. The agent now behaves inconsistently, and there is no timestamp to break the
tie. *Fix:* have the agent **update** existing memory files rather than only appending new ones.

**3. Leakage.** The namespace is wrong, or absent, and one customer's preferences surface in
another's conversation. This is the one that becomes an incident report. *Fix:* make the
namespace a required parameter (it is) and test cross-user isolation explicitly (you just did).

There is a fourth, subtler one: **the agent records something wrong and then trusts it forever.**
Memory has no error correction unless you build some.

### ✍️ Exercise

Teach an agent a formatting rule, then prove it persisted:

1. Build an agent with per-user memory scoping.
2. In thread one, tell it a specific rule (e.g. *"always give me a one-line TL;DR before any
   explanation"*).
3. Inspect the store and read exactly what it wrote.
4. In a **third** thread, ask an unrelated question and confirm the rule is applied.
5. Then have it *change* its mind — tell it the opposite, and check whether the store now
   contains one updated memory or two contradictory ones.

Step 5 is the interesting one.

<details><summary>Show a solution</summary>

```python
from dataclasses import dataclass

@dataclass
class Context:
    user_id: str

store = InMemoryStore()

agent = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You are a helpful assistant.\n"
        "Record durable user preferences under /memories/preferences.md.\n"
        "If a new preference contradicts an existing one, UPDATE the file rather than "
        "adding a second, conflicting note."
    ),
    backend=CompositeBackend(
        default=StateBackend(),
        routes={"/memories/": StoreBackend(
            namespace=lambda rt: ("users", rt.context.user_id, "memories"), store=store)},
    ),
    context_schema=Context,
)

ctx = Context(user_id="avery")

agent.invoke({"messages": [{"role": "user", "content":
    "Always give me a one-line TL;DR before any explanation."}]}, context=ctx)

for item in store.search(("users", "avery", "memories")):
    print(item.key, "->", str(item.value)[:200])

print(agent.invoke({"messages": [{"role": "user", "content":
    "Why do agents need a filesystem?"}]}, context=ctx)["messages"][-1].text[:300])

agent.invoke({"messages": [{"role": "user", "content":
    "Actually, drop the TL;DR. Just answer directly."}]}, context=ctx)

print("\n--- after the reversal ---")
for item in store.search(("users", "avery", "memories")):
    print(item.key, "->", str(item.value)[:300])
```

</details>

---

## 📌 Key takeaways

- "Memory" is three mechanisms with three owners: system prompt (engineers), `AGENTS.md` (the team), `/memories/` (the agent).
- `AGENTS.md` is the tier a non-engineer can change without a deploy — that is its whole point.
- Recall is **retrieval plus prompt assembly**, not the model remembering. That tells you where to debug.
- Agent-written memory needs a namespace, and cross-user isolation must be tested, not assumed.
- Memory that only appends will eventually contradict itself; instruct the agent to update.
- Everything in `/memories/` is paid for on every model call, so hoarding is a cost bug and a quality bug.

---

## ➡️ Next

**[08 · Skills: progressive disclosure of procedures](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/08_skills.ipynb)**

`AGENTS.md` is always loaded. But an agent with a dozen procedures cannot afford to carry all of
them at once — which is what **skills** solve.